# Computation Graph for y = x²

PyTorch builds a **dynamic computation graph** during the forward pass. Each operation becomes a node; edges carry tensors and gradients flow backward along them.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Forward pass: build the graph
`x` is a leaf tensor with `requires_grad=True`. Squaring creates node `PowBackward`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
print(f"x={x.item():.2f}, y={y.item():.2f}")
print(f"y.grad_fn: {y.grad_fn}")

## 2. Visualize nodes and values

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
nodes = ['x\n(3.0)', 'x²\n(9.0)']
xs = [0.2, 0.8]
for i, (node, xp) in enumerate(zip(nodes, xs)):
    ax.add_patch(plt.Circle((xp, 0.5), 0.12, fc='lightblue', ec='navy', lw=2))
    ax.text(xp, 0.5, node, ha='center', va='center', fontsize=11)
ax.annotate('', xy=(0.68, 0.5), xytext=(0.32, 0.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='darkgreen'))
ax.text(0.5, 0.62, 'forward: y = x²', ha='center', fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('Computation graph: one multiply path')
plt.tight_layout(); plt.show()

## 3. Backward pass: gradient dy/dx = 2x

In [ ]:
y.backward()
print(f"dy/dx = {x.grad.item():.2f}  (expected 2×3 = 6)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['x', 'y'], [x.item(), y.item()], color=['steelblue', 'coral'])
axes[0].set_title('Forward values'); axes[0].set_ylabel('value')
axes[1].bar(['∂y/∂x'], [x.grad.item()], color='seagreen')
axes[1].set_title('Backward gradient'); axes[1].set_ylabel('gradient')
plt.tight_layout(); plt.show()

## 4. Chained operations extend the graph
Each intermediate tensor keeps a link to its parent via `.grad_fn`.

In [ ]:
x2 = torch.tensor(2.0, requires_grad=True)
z = (x2 ** 2) + 3 * x2
z.backward()
print(f"z = {z.item():.2f}, dz/dx = {x2.grad.item():.2f}")

fig, ax = plt.subplots(figsize=(9, 3))
labels = ['x²', '+3x', 'z']
vals = [x2.item()**2, 3*x2.item(), z.item()]
ax.plot(labels, vals, 'o-', color='purple', lw=2, markersize=10)
ax.set_title('Intermediate forward values along the chain')
ax.set_ylabel('value'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()